## 依赖与环境
需要已安装并可用：`torch`, `torch_geometric`, `ema-pytorch` 及项目内模块。
请确保工作目录在 `CatIF_RL-main` 及其依赖可导入。

In [ ]:
import os
import random
import numpy as np
import torch
from dataclasses import dataclass
from tqdm import tqdm
from torch_geometric.loader import DataLoader
from diffusion.gradeif_app import EGNN_NET, GraDe_IF
from dataset_src.large_dataset import Cath
from ema_pytorch import EMA

In [ ]:
AMINO_CODES = ['A','R','N','D','C','Q','E','G','H','I',
               'L','K','M','F','P','S','T','W','Y','V']

def onehot_to_seq(onehot):
    idx = onehot.argmax(dim=-1).cpu().tolist()
    return ''.join(AMINO_CODES[i] for i in idx)

def set_seed(seed):
    if seed < 0:
        return
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
def parse_mask_indices(indices_str):
    """解析索引字符串，支持 '1,2,5-8' 格式，返回 0-based 列表。"""
    if not indices_str:
        return []
    indices = set()
    parts = indices_str.split(',')
    for part in parts:
        part = part.strip()
        if '-' in part:
            start, end = map(int, part.split('-'))
            indices.update(range(start, end + 1))
        else:
            if part:
                indices.add(int(part))
    return sorted(list(indices))

def create_batch_mask(batch, user_indices, device):
    """构建全局 Mask：1=Keep(GT), 0=Inpaint(Gen)"""
    total_nodes = batch.x.shape[0]
    mask = torch.ones((total_nodes, 1), device=device)
    if not hasattr(batch, 'ptr'):
        ptr = [0, total_nodes]
    else:
        ptr = batch.ptr.cpu().tolist()
    for i in range(len(ptr) - 1):
        start_node = ptr[i]
        end_node = ptr[i + 1]
        length = end_node - start_node
        for local_idx in user_indices:
            if local_idx < length:
                global_idx = start_node + local_idx
                mask[global_idx] = 0.0
    return mask

In [ ]:
def repaint_ddim_sample(diffusion_model, data, mask, gt_x, step=100, jump_n=10, diverse=True):
    """支持 Resampling (Jump) 的 RePaint 采样函数。"""
    timesteps = diffusion_model.timesteps
    device = data.x.device
    limit_dist = torch.ones(20) / 20
    zt = diffusion_model.sample_discrete_feature_noise(limit_dist=limit_dist, num_node=data.x.shape[0])
    zt = zt.to(device)
    times_list = list(reversed(range(0, timesteps, step)))
    pbar_desc = f"Repaint(jump={jump_n}, div={diverse})"

    for s_int in tqdm(times_list, desc=pbar_desc):
        t_int = s_int + step
        s_array = s_int * torch.ones((data.batch[-1] + 1, 1)).to(device)
        t_array = t_int * torch.ones((data.batch[-1] + 1, 1)).to(device)
        s_norm = s_array / timesteps
        t_norm = t_array / timesteps
        current_jump_n = 1 if s_int == 0 else jump_n

        for u in range(current_jump_n):
            is_last_step = (s_int == 0) and (u == current_jump_n - 1)
            zt_pred, final_predicted_X = diffusion_model.sample_p_zs_given_zt(
                t_norm, s_norm, zt, data, cond=False, diverse=diverse, step=step, last_step=is_last_step
            )
            s_int_batch = torch.full((data.batch[-1] + 1, 1), s_int, device=device).float()
            temp_data = data.clone()
            temp_data.x = gt_x
            zt_known = diffusion_model.apply_noise(temp_data, s_int_batch).x
            zt = mask * zt_known + (1 - mask) * zt_pred

            if u < current_jump_n - 1 and s_int > 0:
                alpha_t_bar = diffusion_model.noise_schedule.get_alpha_bar(t_normalized=t_norm)
                alpha_s_bar = diffusion_model.noise_schedule.get_alpha_bar(t_normalized=s_norm)
                alpha_rel = (alpha_t_bar / alpha_s_bar).clamp(0, 1)
                if diffusion_model.config['noise_type'] == 'uniform':
                    Qt_jump = diffusion_model.transition_model.get_Qt_bar(alpha_rel, device=device)
                    prob_jump = (Qt_jump[data.batch] @ zt.unsqueeze(2)).squeeze()
                    zt_idx = prob_jump.multinomial(1).squeeze()
                    zt = torch.nn.functional.one_hot(zt_idx, num_classes=20).float()
                else:
                    pass
    return zt, final_predicted_X

In [ ]:
@dataclass
class InferenceConfig:
    test_dir: str
    ckpt_path: str
    output_dir: str
    batch_size: int = 32
    device: str = 'cuda:0'
    seed: int = -1
    step: int = 50
    diverse: bool = True
    use_repaint: bool = False
    mask_indices: str = ''
    repaint_jump_n: int = 10

def load_diffusion_model(ckpt_path, device):
    print(f'Loading checkpoint from {ckpt_path} ...')
    ckpt = torch.load(ckpt_path, map_location=device)
    if 'config' in ckpt:
        config = ckpt['config']
    elif 'meta' in ckpt:
        config = ckpt['meta']['config']
    else:
        config = ckpt.get('args', {})
        if not config:
            print('Warning: Config not found in ckpt, using defaults might fail.')

    base_model = EGNN_NET(
        input_feat_dim = config.get('input_feat_dim', 0),
        hidden_channels= config['hidden_dim'],
        edge_attr_dim  = config.get('edge_attr_dim', 0),
        dropout        = config['drop_out'],
        n_layers       = config['depth'],
        update_edge    = config.get('update_edge', True),
        embedding      = config.get('embedding', False),
        embedding_dim  = config.get('embedding_dim', 64),
        norm_feat      = config.get('norm_feat', False),
        embed_ss       = config.get('embed_ss', -1),
    )

    diffusion = GraDe_IF(
        model      = base_model,
        timesteps  = config['timesteps'],
        objective  = config.get('objective', 'pred_x0'),
        config     = config
    ).to(device)

    if 'ema' in ckpt:
        ema = EMA(diffusion)
        ema.load_state_dict(ckpt['ema'])
        model = ema.ema_model.to(device).eval()
    elif 'model' in ckpt:
        diffusion.load_state_dict(ckpt['model'], strict=False)
        model = diffusion.to(device).eval()
    else:
        diffusion.load_state_dict(ckpt, strict=False)
        model = diffusion.to(device).eval()

    return model, base_model, config

def run_inference(cfg: InferenceConfig):
    device = torch.device(cfg.device if torch.cuda.is_available() else 'cpu')
    set_seed(cfg.seed)
    os.makedirs(cfg.output_dir, exist_ok=True)
    is_diverse = cfg.diverse

    test_ids = sorted([f for f in os.listdir(cfg.test_dir) if f.endswith('.pt')])
    test_ds  = Cath(test_ids, cfg.test_dir)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, pin_memory=True, num_workers=4)

    model, base_model, config = load_diffusion_model(cfg.ckpt_path, device)

    if cfg.use_repaint:
        indices = parse_mask_indices(cfg.mask_indices)
        print(f'🎨 Mode: Repaint | Jump N: {cfg.repaint_jump_n} | Diverse: {is_diverse}')
        print(f'📍 Masked Indices (Inpainting): {indices}')
    else:
        print(f'🚀 Mode: Standard Generation | Diverse: {is_diverse}')

    global_idx = 0
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            if base_model.lin.in_features != batch.x.shape[1] + batch.extra_x.shape[1]:
                pass

            if cfg.use_repaint:
                gt_x = batch.x.clone()
                user_mask_indices = parse_mask_indices(cfg.mask_indices)
                mask = create_batch_mask(batch, user_mask_indices, device)
                zt, pred_onehot = repaint_ddim_sample(
                    model, batch, mask, gt_x, step=cfg.step, jump_n=cfg.repaint_jump_n, diverse=is_diverse
                )
            else:
                zt, pred_onehot = model.ddim_sample(batch, diverse=is_diverse, step=cfg.step)

            node_counts = [g.x.size(0) for g in batch.to_data_list()]
            start = 0
            for count in node_counts:
                if global_idx >= len(test_ids):
                    break
                oh = pred_onehot[start:start + count]
                seq = onehot_to_seq(oh)
                pt_name = test_ids[global_idx]
                fasta_fn = os.path.splitext(pt_name)[0] + '.fasta'
                out_path = os.path.join(cfg.output_dir, fasta_fn)
                with open(out_path, 'w') as f:
                    f.write(f'>{pt_name}\n{seq}\n')
                start += count
                global_idx += 1
    print(f'✅ All Done. Output saved to {cfg.output_dir}')

## 使用示例
将路径替换为你本地的测试集与 checkpoint。
mask 索引是 0-based（与脚本一致）。

In [ ]:
cfg = InferenceConfig(
    test_dir='dataset/process/test',
    ckpt_path='diffusion/results/your_ckpt.pt',
    output_dir='sampling/output',
    batch_size=32,
    device='cuda:0',
    seed=1,
    step=50,
    diverse=True,
    use_repaint=True,
    mask_indices='5,10-20',
    repaint_jump_n=1,
)
# 取消注释后运行
# run_inference(cfg)